In [1]:
import random
import math

import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from mpl_finance import candlestick_ohlc
import matplotlib.dates as mdates

from binance import Client

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical

/opt/homebrew/Caskroom/miniforge/base/envs/coin/lib/python3.9/site-packages/mpl_finance.py:16: DeprecationWarning: 



    Please use `mplfinance` instead (no hyphen, no underscore).

    To install: `pip install --upgrade mplfinance` 

   For more information, see: https://pypi.org/project/mplfinance/


  __warnings.warn('\n\n  ================================================================='+


In [2]:
# Hyperparameters
learning_rate = 0.0002
gamma = 0.95

In [3]:
client = Client()
klines = np.array(client.get_historical_klines("ETHUSDT", Client.KLINE_INTERVAL_1HOUR, "1 Jan, 2018"))
sample = pd.DataFrame(klines.reshape(-1, 12), dtype=float, columns=['datetime',
                                                                   'open',
                                                                   'high',
                                                                   'low',
                                                                   'close',
                                                                   'volume',
                                                                   'close time',
                                                                   'quote asset volume, number of trades',
                                                                   'number of trades',
                                                                   'taker buy base asset volume',
                                                                   'taker buy quote asset volume',
                                                                   'ignore'])
sample['datetime'] = pd.to_datetime(sample['datetime'], unit='ms')
sample.set_index('datetime', inplace=True)
sample = sample[['open', 'high', 'low', 'close', 'volume']]

In [4]:
# make states
sample['ropen_7'] = (sample['open'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rhigh_7'] = (sample['high'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rlow_7'] = (sample['low'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rclose_7'] = (sample['close'] - sample['low'].rolling(7).min())/(sample['high'].rolling(7).max() - sample['low'].rolling(7).min())
sample['rvolume_7'] = (sample['volume'] - sample['volume'].rolling(7).min())/(sample['volume'].rolling(7).max() - sample['volume'].rolling(7).min())

sample['ropen_14'] = (sample['open'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rhigh_14'] = (sample['high'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rlow_14'] = (sample['low'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rclose_14'] = (sample['close'] - sample['low'].rolling(14).min())/(sample['high'].rolling(14).max() - sample['low'].rolling(14).min())
sample['rvolume_14'] = (sample['volume'] - sample['volume'].rolling(14).min())/(sample['volume'].rolling(14).max() - sample['volume'].rolling(14).min())

sample['ropen_24'] = (sample['open'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rhigh_24'] = (sample['high'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rlow_24'] = (sample['low'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rclose_24'] = (sample['close'] - sample['low'].rolling(24).min())/(sample['high'].rolling(24).max() - sample['low'].rolling(24).min())
sample['rvolume_24'] = (sample['volume'] - sample['volume'].rolling(24).min())/(sample['volume'].rolling(24).max() - sample['volume'].rolling(24).min())

sample['ropen_168'] = (sample['open'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rhigh_168'] = (sample['high'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rlow_168'] = (sample['low'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rclose_168'] = (sample['close'] - sample['low'].rolling(168).min())/(sample['high'].rolling(168).max() - sample['low'].rolling(168).min())
sample['rvolume_168'] = (sample['volume'] - sample['volume'].rolling(168).min())/(sample['volume'].rolling(168).max() - sample['volume'].rolling(168).min())

sample['ropen_1500'] = (sample['open'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rhigh_1500'] = (sample['high'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rlow_1500'] = (sample['low'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rclose_1500'] = (sample['close'] - sample['low'].rolling(1500).min())/(sample['high'].rolling(1500).max() - sample['low'].rolling(1500).min())
sample['rvolume_1500'] = (sample['volume'] - sample['volume'].rolling(1500).min())/(sample['volume'].rolling(1500).max() - sample['volume'].rolling(1500).min())

sample.dropna(inplace=True)

In [5]:
cut1 = int(len(sample)*0.7)
cut2 = int(len(sample)*0.8)
train_sample, val_sample, test_sample = sample.iloc[:cut1], sample.iloc[cut1:cut2], sample.iloc[cut2:]

In [6]:
class Environment():
    def __init__(self, sample):
        self.sample = sample.copy()
        self.episode = None
        self.epi_idx = 0
        
    def make_episode(self, start_idx=-1):
        if start_idx > len(self.sample) - 2000:
            print(f"start_idx should be smaller than {len(self.sample)-2000}")
            raise(ValueError)
        if start_idx == -1: start_idx = random.randint(0, len(self.sample)-2000)
        
        self.episode = self.sample[start_idx:start_idx+2000]
        self.epi_idx = 0
        first_state = self.episode.iloc[self.epi_idx].to_numpy()[5:]
        return first_state
    
    def step(self, action):
        this_close = self.episode.iloc[self.epi_idx].to_numpy()[3]
    
        self.epi_idx += 1
        
        # is done?
        done = True if self.epi_idx == 2000-1 else False
        
        #print(len(self.episode), self.epi_idx)
        # make state
        state = self.episode.iloc[self.epi_idx].to_numpy()[5:]
        
        # make reward
        next_high = self.episode.iloc[self.epi_idx].to_numpy()[1]
        next_low = self.episode.iloc[self.epi_idx].to_numpy()[2]
        reward = np.log(random.uniform(next_low, next_high)/this_close)
        
        if action == 0: reward = 0.
        
        # index and ohlcv
        info = (self.epi_idx, self.episode.iloc[self.epi_idx].to_numpy()[:5])
        
        return (state, reward, done, info)

In [7]:
class Policy(nn.Module):
    def __init__(self):
        super(Policy, self).__init__()
        self.data = []
        
        self.fc1 = nn.Linear(25, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 2)
        self.optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        
        # history
        self.episodes = 0
        self.scores = list()
        self.loss = list()
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=0)
        return x
    
    def put_data(self,item):
        self.data.append(item)
    
    def train_net(self):
        R = 0.
        self.optimizer.zero_grad()
        for r, prob in self.data[::-1]:
            R = r + gamma*R
            loss = -R*torch.log(prob)
            loss.backward()
        self.optimizer.step()
        self.data = []
        
        return loss.detach().numpy()
    

In [8]:
class History():
    def __init__(self):
        self.episodes = 0
        self.score = []
        self.loss = []
    
    def update(self, score, loss):
        self.episodes += 1
        self.score.append(score)
        self.loss.append(loss)
    
    def plot(self):
        plt.figure(figsize=(15, 8))
        plt.subplot(2, 1, 1)
        plt.plot(range(self.episodes), self.score, 'r--', label='score')
        plt.legend(loc='best')
        
        plt.subplot(2, 1, 2)
        plt.plot(range(self.episodes), self.loss, 'b--', label='loss')
        plt.legend(loc='best')
        
        plt.show()

In [10]:
env = Environment(train_sample)
pi = Policy()
history = History()
acc_score = 0.
print_interval = 20
estimate_interval = 100

for n_epi in range(4000):
    score = 0.
    state = env.make_episode()
    done = False
    
    while not done:
        prob = pi(torch.from_numpy(state).float())
        m = Categorical(prob)
        action = m.sample()
        next_state, reward, done, info = env.step(action.item())
        pi.put_data((reward, prob[action]))
        state = next_state
        score += reward
    loss = pi.train_net()
    history.update(score, loss)
    
    acc_score += score
    if n_epi%print_interval == 0 and n_epi != 0:
        print(f"[EPISODE {n_epi}] avg return: {(np.exp(acc_score/print_interval)-1)*100:.2f}%")
        acc_score = 0.

history.plot()

[EPISODE 20] avg return: -14.89%
[EPISODE 40] avg return: -16.35%
[EPISODE 60] avg return: -17.15%
[EPISODE 80] avg return: -12.59%
[EPISODE 100] avg return: -16.40%
[EPISODE 120] avg return: -16.48%
[EPISODE 140] avg return: -12.94%
[EPISODE 160] avg return: -13.17%
[EPISODE 180] avg return: -15.42%
[EPISODE 200] avg return: -17.38%
[EPISODE 220] avg return: -17.02%
[EPISODE 240] avg return: -9.11%
[EPISODE 260] avg return: -2.83%
[EPISODE 280] avg return: -13.17%
[EPISODE 300] avg return: -12.06%
[EPISODE 320] avg return: -3.69%
[EPISODE 340] avg return: -2.38%
[EPISODE 360] avg return: -3.19%
[EPISODE 380] avg return: -1.06%
[EPISODE 400] avg return: -3.41%
[EPISODE 420] avg return: -0.18%
[EPISODE 440] avg return: -1.97%
[EPISODE 460] avg return: -4.95%
[EPISODE 480] avg return: -5.13%
[EPISODE 500] avg return: 0.20%
[EPISODE 520] avg return: -2.39%
[EPISODE 540] avg return: -3.56%
[EPISODE 560] avg return: 1.07%
[EPISODE 580] avg return: -2.11%
[EPISODE 600] avg return: -1.54%
[EP

In [ ]:
train_states = train_sample.to_numpy()

#  Buy and hold
book = train_sample[['close']].copy()
book['number'] = book.index.map(mdates.date2num)
book['reward'] = 1.
book['total_reward'] = 1.
book['action'] = False
for i, idx in enumerate(book.index):
    state = train_states[i][5:]
    prob = pi(torch.from_numpy(state).float())
    m = Categorical(prob)
    action = m.sample().item()
    if action == 0:
        book.loc[idx, 'reward'] = 1.
    else:
        book.loc[idx, 'action'] = True
        book.loc[idx, 'reward'] = train_sample.shift(-1).loc[idx, 'close']/train_sample.loc[idx, 'close']

total_reward = 1. 
for idx in book.index:
    total_reward *= book.loc[idx, 'reward']
    book.loc[idx, 'total_reward'] = total_reward
CAGR = book['total_reward'].iloc[-1]**(365*24/len(book.index)) - 1.
historical_max = book['total_reward'].cummax()
daily_drawdown = book['total_reward']/historical_max - 1.
historical_dd = daily_drawdown.cummin()
MDD = historical_dd.min()
VOL = np.std(book['reward'])*np.sqrt(365.*24)
Sharpe = (np.mean(book['reward'])/np.std(book['reward'])*np.sqrt(365.*24))

print("==== REINFORCE trading ====")
print("Accumulated Returns:", round((total_reward-1.)*100, 2), "%")
print("CAGR:", round(CAGR*100, 2), "%")
print("MDD:", round(MDD*100, 2), "%")
print("VOL:", round(VOL*100, 3), "%")
print("Sharpe:", round(Sharpe*100,2), "%")

NameError: name 'pi' is not defined